# 🚦 Next-Hour Traffic Volume Prediction

### A machine-learning study using the Metro Interstate Traffic Volume dataset

**Project presentation notebook**

| Project detail | |
|---|---|
| Project domain | Machine Learning · Intelligent Transportation |
| Problem type | Supervised regression |
| Dataset | Metro Interstate Traffic Volume |
| Study location | Westbound I-94, Minneapolis–St Paul, Minnesota |
| Team members | *Add names and roll numbers* |
| Faculty guide | *Add professor's name* |

---

## Project overview

Traffic conditions change throughout the day. Understanding how traffic volume is likely to change can support planning and monitoring. In this project, we build and evaluate machine-learning models that estimate **the number of vehicles in the next hour** from the current traffic observation, weather conditions, and known calendar information.

The notebook follows a complete project workflow: understand the data, explore patterns, prepare hourly observations, create a future prediction target, train models, compare them with simple baselines, and interpret the results.

> **Scope of the project:** The dataset contains vehicle counts, not official congestion labels, road speeds, or road density. Therefore, this project predicts **traffic volume**. It does not claim to classify congestion levels.

## 1. Problem statement and objectives

### Problem statement

Given the traffic volume and weather observations for the current hour, along with calendar information for the upcoming hour, estimate how many vehicles will be recorded in that upcoming hour.

### Objectives

1. Explore historical traffic volume and its relationship with time and weather.
2. Clean repeated timestamps and prepare a consistent hourly dataset.
3. Formulate next-hour traffic prediction as a supervised regression task.
4. Train a Decision Tree Regressor and a Random Forest Regressor.
5. Compare both models against simple prediction baselines.
6. Evaluate prediction errors and identify the features most useful to the fitted model.

### Research question

**Can a machine-learning model estimate next-hour traffic volume more accurately than simply carrying forward the current hour's traffic count?**

## 2. Dataset description

The project uses the **Metro Interstate Traffic Volume** dataset from the UCI Machine Learning Repository. It contains historical hourly traffic counts for westbound I-94 near Minneapolis–St Paul, along with weather and holiday information.

| Field | Meaning | Use in this project |
|---|---|---|
| `date_time` | Date and time of the observation | Ordering observations |
| `traffic_volume` | Number of vehicles recorded in the hour | Current input and future target |
| `temp` | Temperature in Kelvin | Converted to Celsius |
| `rain_1h` | Rain measured during the hour (mm) | Weather input |
| `snow_1h` | Snow measured during the hour (mm) | Weather input |
| `clouds_all` | Cloud-cover percentage | Weather input |
| `weather_main` | General weather category | Categorical input |
| `holiday` | Holiday label | Calendar input |

**Dataset source:** Hogue, J. (2019). *Metro Interstate Traffic Volume* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5X60B.

The dataset is licensed under **CC BY 4.0**. The CSV must be attached to this Kaggle notebook through **Add Input** before running the cells.

## 3. Import libraries and locate the Kaggle dataset

The notebook searches Kaggle's read-only `/kaggle/input` directory for a CSV containing the expected dataset columns. It does not depend on a particular Kaggle dataset folder name.

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)

required_columns = {
    "holiday", "temp", "rain_1h", "snow_1h", "clouds_all",
    "weather_main", "date_time", "traffic_volume"
}

candidate_files = []
for pattern in ("/kaggle/input/**/*.csv", "/kaggle/input/**/*.csv.gz"):
    candidate_files.extend(glob.glob(pattern, recursive=True))

matching_files = []
for file_path in candidate_files:
    try:
        columns = set(pd.read_csv(file_path, nrows=0).columns)
        if required_columns.issubset(columns):
            matching_files.append(file_path)
    except Exception:
        continue

if not matching_files:
    print("CSV files found under /kaggle/input:")
    for file_path in candidate_files:
        print(" -", file_path)
    raise FileNotFoundError(
        "The Metro Interstate Traffic Volume CSV was not found. "
        "Use Kaggle's Add Input option to attach the dataset, then run this cell again."
    )

csv_path = matching_files[0]
if len(matching_files) > 1:
    print("Multiple matching files found. Using:", csv_path)

raw = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded dataset: {csv_path}")
print(f"Dataset shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
display(raw.head())

## 4. Initial data inspection

Before modelling, we inspect the column types, missing values, duplicate rows, and date coverage. This helps identify data-quality issues that could affect later steps.

In [ ]:
print("Column names:")
print(raw.columns.tolist())

print("\nDataset information:")
raw.info()

quality = pd.DataFrame({
    "Missing values": raw.isna().sum(),
    "Missing (%)": (raw.isna().mean() * 100).round(2),
})
display(quality.sort_values("Missing (%)", ascending=False))

print(f"Exact duplicate rows: {raw.duplicated().sum():,}")

raw["date_time"] = pd.to_datetime(raw["date_time"], errors="coerce")
raw["traffic_volume"] = pd.to_numeric(raw["traffic_volume"], errors="coerce")

print("Invalid timestamps:", raw["date_time"].isna().sum())
print("Date range:", raw["date_time"].min(), "to", raw["date_time"].max())
print("Unique timestamps:", raw["date_time"].nunique())

## 5. Exploratory data analysis

Exploratory Data Analysis (EDA) helps us understand the target variable before building a model.

We first examine the distribution of hourly vehicle counts, then compare average traffic volume across hours of the day. These patterns help determine whether calendar information may be useful for prediction.

In [ ]:
eda_data = raw.dropna(subset=["date_time", "traffic_volume"]).copy()

print("Traffic volume summary:")
display(eda_data["traffic_volume"].describe().to_frame().round(2))

plt.figure(figsize=(10, 5))
plt.hist(eda_data["traffic_volume"], bins=40, edgecolor="white")
plt.title("Distribution of Hourly Traffic Volume")
plt.xlabel("Vehicles recorded per hour")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

In [ ]:
eda_data["hour"] = eda_data["date_time"].dt.hour
hourly_profile = eda_data.groupby("hour")["traffic_volume"].mean()

plt.figure(figsize=(10, 5))
plt.plot(hourly_profile.index, hourly_profile.values, marker="o", linewidth=2)
plt.title("Average Traffic Volume by Hour of Day")
plt.xlabel("Hour of day")
plt.ylabel("Average vehicles per hour")
plt.xticks(range(0, 24))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Traffic patterns across weekdays and hours

A weekday-hour profile can reveal recurring daily patterns. The chart below displays the average traffic volume for each weekday and hour. It is descriptive analysis, not evidence that the calendar variables cause the observed traffic changes.

In [ ]:
weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
eda_data["weekday"] = eda_data["date_time"].dt.dayofweek

weekday_hour = eda_data.pivot_table(
    index="weekday",
    columns="hour",
    values="traffic_volume",
    aggfunc="mean"
).reindex(index=range(7), columns=range(24))

plt.figure(figsize=(13, 5))
image = plt.imshow(weekday_hour, aspect="auto", interpolation="nearest")
plt.colorbar(image, label="Average vehicles per hour")
plt.title("Average Traffic Volume by Weekday and Hour")
plt.xlabel("Hour of day")
plt.ylabel("Day of week")
plt.xticks(range(24))
plt.yticks(range(7), weekday_names)
plt.tight_layout()
plt.show()

### Traffic volume by weather category

We also inspect average vehicle counts across the most frequently observed weather categories. Categories with very few records are excluded from this chart so that small groups do not dominate the interpretation.

In [ ]:
weather_summary = (
    eda_data.groupby("weather_main")["traffic_volume"]
    .agg(["mean", "count"])
    .query("count >= 50")
    .sort_values("mean", ascending=False)
)

display(weather_summary.round(1))

plt.figure(figsize=(10, 5))
plt.bar(weather_summary.index.astype(str), weather_summary["mean"])
plt.title("Average Traffic Volume by Weather Category")
plt.xlabel("Weather category")
plt.ylabel("Average vehicles per hour")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. Data cleaning and hourly consolidation

The source can contain multiple rows with the same timestamp. A next-hour model needs one consistent record per timestamp, so we consolidate repeated timestamps.

- Numeric weather measurements and traffic volume are combined using the median.
- Holiday and weather categories are combined using the most frequent non-missing value.
- Rows without a valid timestamp or traffic-volume target are removed.
- Temperature is converted from Kelvin to Celsius.

Using the median makes the aggregation less sensitive to unusual duplicate measurements. We also report how many timestamps had conflicting traffic-volume values.

In [ ]:
work = raw.copy()

numeric_columns = ["temp", "rain_1h", "snow_1h", "clouds_all", "traffic_volume"]
for column in numeric_columns:
    work[column] = pd.to_numeric(work[column], errors="coerce")

work = work.dropna(subset=["date_time", "traffic_volume"]).copy()
work["holiday"] = work["holiday"].fillna("None").astype(str)
work["weather_main"] = work["weather_main"].fillna("Unknown").astype(str)

volume_counts = work.groupby("date_time")["traffic_volume"].nunique()
conflicting_timestamps = int((volume_counts > 1).sum())

def most_common_value(series):
    values = series.dropna().astype(str)
    if values.empty:
        return "Unknown"
    return values.value_counts().index[0]

aggregation = {
    "holiday": most_common_value,
    "temp": "median",
    "rain_1h": "median",
    "snow_1h": "median",
    "clouds_all": "median",
    "weather_main": most_common_value,
    "traffic_volume": "median",
}

traffic = (
    work.groupby("date_time", as_index=False)
    .agg(aggregation)
    .sort_values("date_time")
    .reset_index(drop=True)
)

traffic["traffic_volume"] = traffic["traffic_volume"].round().astype(int)
traffic["temp_c"] = traffic["temp"] - 273.15

print(f"Rows after removing invalid target/timestamp: {len(work):,}")
print(f"Unique hourly timestamps after consolidation: {len(traffic):,}")
print(f"Timestamps with conflicting traffic counts: {conflicting_timestamps:,}")
print(f"Duplicate timestamps remaining: {traffic['date_time'].duplicated().sum():,}")
display(traffic.head())

## 7. Create the next-hour prediction target

This step converts the historical time series into a supervised learning dataset.

For an observation at time **t**:

- Inputs include the traffic volume and weather observed at time *t*.
- The target is the traffic volume recorded at time *t + 1 hour*.
- Calendar features are calculated from the timestamp being predicted.

We keep only pairs exactly one hour apart. This prevents the model from treating a two-hour or longer gap as a one-hour prediction.

In [ ]:
traffic = traffic.sort_values("date_time").reset_index(drop=True).copy()

traffic["next_time"] = traffic["date_time"].shift(-1)
traffic["target_next_hour"] = traffic["traffic_volume"].shift(-1)
traffic["future_holiday"] = traffic["holiday"].shift(-1)

consecutive_hours = traffic["next_time"].eq(
    traffic["date_time"] + pd.Timedelta(hours=1)
)

samples = traffic.loc[consecutive_hours].copy()
samples["future_hour"] = samples["next_time"].dt.hour
samples["future_dayofweek"] = samples["next_time"].dt.dayofweek
samples["future_month"] = samples["next_time"].dt.month
samples["future_is_weekend"] = (samples["future_dayofweek"] >= 5).astype(int)

samples = samples.sort_values("next_time").reset_index(drop=True)

print(f"Consolidated hourly records: {len(traffic):,}")
print(f"Valid next-hour examples: {len(samples):,}")
print(f"Records excluded due to a missing/non-consecutive next hour: {len(traffic) - len(samples):,}")

display(samples[[
    "date_time", "next_time", "traffic_volume",
    "target_next_hour", "future_hour", "future_dayofweek"
]].head(8))

## 8. Define model inputs and create a time-based split

The target is `target_next_hour`.

### Input features

**Current observations:** current traffic volume, temperature, rain, snow, cloud cover, and current weather category.

**Known future calendar information:** hour, weekday, month, weekend indicator, and holiday label for the hour being predicted.

We use a chronological 80/20 split. Earlier observations train the models, and the latest observations are held out for testing. This better reflects the real task of predicting a future period than randomly shuffling time-series rows.

In [ ]:
numeric_features = [
    "traffic_volume",
    "temp_c",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "future_hour",
    "future_dayofweek",
    "future_month",
    "future_is_weekend",
]

categorical_features = ["weather_main", "future_holiday"]
features = numeric_features + categorical_features
target = "target_next_hour"

model_data = samples.dropna(subset=[target]).copy()
split_index = int(len(model_data) * 0.80)

train = model_data.iloc[:split_index].copy()
test = model_data.iloc[split_index:].copy()

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Training examples: {len(train):,}")
print(f"Testing examples: {len(test):,}")
print("Training period:", train["next_time"].min(), "to", train["next_time"].max())
print("Testing period: ", test["next_time"].min(), "to", test["next_time"].max())

if len(train) < 100 or len(test) < 20:
    raise ValueError("There are too few valid examples. Check that the correct dataset is attached.")

## 9. Establish a simple baseline

A machine-learning model should be compared against a sensible simple approach.

Our first baseline is the **persistence baseline**:

> Predict that the next hour's traffic volume will be the same as the current hour's traffic volume.

This approach requires no machine learning. It gives us a practical reference point: the trained model should ideally reduce prediction error compared with this simple rule.

In [ ]:
def regression_metrics(model_name, actual, predicted):
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }

persistence_predictions = X_test["traffic_volume"].to_numpy()
persistence_metrics = regression_metrics(
    "Persistence baseline", y_test, persistence_predictions
)

display(pd.DataFrame([persistence_metrics]).round(3))

## 10. Build and train the regression models

We compare two models:

1. **Decision Tree Regressor:** learns a sequence of feature-based splits. It is relatively easy to explain, but a single tree can be sensitive to the training data.
2. **Random Forest Regressor:** combines predictions from many decision trees. Averaging the trees can produce more stable predictions.

Preprocessing is included inside each model pipeline:
- Missing numeric values are replaced with the training median.
- Missing categorical values are replaced with the most frequent training category.
- Categorical values are one-hot encoded.
- Unknown categories at test time are handled safely.

Because preprocessing is fitted within the pipeline, the test set is not used to fit the transformations.

In [ ]:
def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols),
    ])

model_definitions = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=20,
        random_state=42,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
}

trained_models = {}
test_predictions = {}

for model_name, estimator in model_definitions.items():
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor(numeric_features, categorical_features)),
        ("model", estimator),
    ])

    pipeline.fit(X_train, y_train)
    test_predictions[model_name] = pipeline.predict(X_test)
    trained_models[model_name] = pipeline

    print(f"{model_name} training completed.")

## 11. Evaluate the models

We use three regression metrics:

- **MAE (Mean Absolute Error):** average absolute prediction error, measured in vehicles per hour. Lower is better.
- **RMSE (Root Mean Squared Error):** penalises larger prediction errors more heavily. Lower is better.
- **R² (coefficient of determination):** compares the model's predictions with a mean-value reference. A value closer to 1 generally indicates a better fit. It is **not** a classification accuracy percentage.

All models are evaluated on the same chronological test period.

In [ ]:
evaluation_rows = [persistence_metrics]

for model_name, predicted in test_predictions.items():
    evaluation_rows.append(
        regression_metrics(model_name, y_test, predicted)
    )

metrics_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("Performance on the held-out future period:")
display(metrics_df.round(3))

baseline_mae = metrics_df.loc[
    metrics_df["Model"] == "Persistence baseline", "MAE"
].iloc[0]

print("\nMAE change relative to the persistence baseline:")
for _, row in metrics_df.iterrows():
    if row["Model"] != "Persistence baseline":
        change = (baseline_mae - row["MAE"]) / baseline_mae * 100
        direction = "lower" if change >= 0 else "higher"
        print(f"{row['Model']}: {abs(change):.2f}% {direction} MAE")

## 12. Compare against the same-hour-yesterday baseline

Traffic often follows daily routines. We therefore add a second baseline: use the traffic volume from the **same hour on the previous day** as the prediction.

This comparison is performed only for test timestamps where the previous-day observation exists. Every row in this comparison therefore uses the same target timestamps.

In [ ]:
volume_by_time = (
    traffic.drop_duplicates("date_time")
    .set_index("date_time")["traffic_volume"]
)

comparison = test[[
    "next_time", "target_next_hour", "traffic_volume"
]].copy().reset_index(drop=True)

comparison["previous_hour_prediction"] = comparison["traffic_volume"]
comparison["yesterday_prediction"] = (
    comparison["next_time"] - pd.Timedelta(hours=24)
).map(volume_by_time)

comparison["decision_tree_prediction"] = test_predictions["Decision Tree"]
comparison["random_forest_prediction"] = test_predictions["Random Forest"]

comparison = comparison.dropna(subset=["yesterday_prediction"]).copy()

comparison_rows = []
comparison_methods = [
    ("Previous-hour baseline", "previous_hour_prediction"),
    ("Same-hour-yesterday baseline", "yesterday_prediction"),
    ("Decision Tree", "decision_tree_prediction"),
    ("Random Forest", "random_forest_prediction"),
]

for model_name, prediction_column in comparison_methods:
    comparison_rows.append(
        regression_metrics(
            model_name,
            comparison["target_next_hour"],
            comparison[prediction_column],
        )
    )

comparison_metrics = (
    pd.DataFrame(comparison_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print(f"Common test observations used: {len(comparison):,}")
display(comparison_metrics.round(3))

## 13. Visualise actual and predicted traffic volume

A metric table summarises errors, but a time-series plot helps us see whether predictions follow changes in traffic volume.

The chart compares actual test observations with Random Forest predictions over the first seven days of the held-out period (or the available test data if fewer observations exist).

In [ ]:
prediction_plot = test[["next_time", "target_next_hour"]].copy().reset_index(drop=True)
prediction_plot["random_forest_prediction"] = test_predictions["Random Forest"]
prediction_plot = prediction_plot.sort_values("next_time").head(24 * 7)

plt.figure(figsize=(14, 5))
plt.plot(
    prediction_plot["next_time"],
    prediction_plot["target_next_hour"],
    label="Actual traffic volume",
    linewidth=2,
)
plt.plot(
    prediction_plot["next_time"],
    prediction_plot["random_forest_prediction"],
    label="Random Forest prediction",
    linewidth=2,
)
plt.title("Actual vs Predicted Next-Hour Traffic Volume")
plt.xlabel("Prediction time")
plt.ylabel("Vehicles per hour")
plt.legend()
plt.xticks(rotation=30)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 14. Examine prediction errors

We inspect the prediction errors rather than looking only at the best-performing examples.

The residual is calculated as:

\[
\text{Residual} = \text{Actual volume} - \text{Predicted volume}
\]

A positive residual means the model underpredicted traffic volume; a negative residual means it overpredicted.

In [ ]:
prediction_results = test[[
    "next_time", "target_next_hour", "traffic_volume", "future_hour"
]].copy().reset_index(drop=True)

prediction_results["predicted_volume"] = test_predictions["Random Forest"]
prediction_results["absolute_error"] = (
    prediction_results["target_next_hour"]
    - prediction_results["predicted_volume"]
).abs()
prediction_results["residual"] = (
    prediction_results["target_next_hour"]
    - prediction_results["predicted_volume"]
)

display(
    prediction_results[[
        "next_time", "traffic_volume", "target_next_hour",
        "predicted_volume", "absolute_error"
    ]].head(10).round(1)
)

print("Largest absolute prediction errors:")
display(
    prediction_results.nlargest(10, "absolute_error")[[
        "next_time", "traffic_volume", "target_next_hour",
        "predicted_volume", "absolute_error"
    ]].round(1)
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(prediction_results["residual"], bins=40, edgecolor="white")
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Random Forest Prediction Residuals")
plt.xlabel("Actual minus predicted vehicles")
plt.ylabel("Number of test observations")
plt.tight_layout()
plt.show()

In [ ]:
error_by_hour = (
    prediction_results.groupby("future_hour")["absolute_error"]
    .mean()
    .reindex(range(24))
)

plt.figure(figsize=(10, 5))
plt.bar(error_by_hour.index, error_by_hour.values)
plt.title("Average Absolute Prediction Error by Hour")
plt.xlabel("Hour being predicted")
plt.ylabel("Mean absolute error (vehicles)")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

## 15. Understand which features influenced the Random Forest

Feature importance estimates how useful each input was to the fitted Random Forest when constructing its trees.

It can help us understand the model, but it does **not** prove that a feature causes traffic volume to change. Correlated inputs can also share or mask importance.

In [ ]:
random_forest_pipeline = trained_models["Random Forest"]
encoded_feature_names = (
    random_forest_pipeline.named_steps["preprocessor"].get_feature_names_out()
)
feature_importances = (
    random_forest_pipeline.named_steps["model"].feature_importances_
)

importance_df = (
    pd.DataFrame({
        "Feature": encoded_feature_names,
        "Importance": feature_importances,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 features:")
display(importance_df.head(15).round(4))

top_features = importance_df.head(12).sort_values("Importance")

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Most Important Random Forest Features")
plt.xlabel("Feature importance")
plt.ylabel("Encoded input feature")
plt.tight_layout()
plt.show()

## 16. Additional experiment: remove current traffic volume

Current-hour traffic volume is likely to be an informative input for next-hour traffic. To examine its contribution, we train a second Random Forest without that feature.

This is called an **ablation experiment**: remove one input and compare the test performance. The experiment helps show whether weather and calendar information alone provide a similar level of predictive power.

In [ ]:
numeric_without_current_volume = [
    column for column in numeric_features if column != "traffic_volume"
]
features_without_current_volume = (
    numeric_without_current_volume + categorical_features
)

rf_without_volume = Pipeline([
    (
        "preprocessor",
        build_preprocessor(
            numeric_without_current_volume,
            categorical_features,
        ),
    ),
    (
        "model",
        RandomForestRegressor(
            n_estimators=150,
            max_depth=20,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf_without_volume.fit(
    train[features_without_current_volume],
    train[target],
)

prediction_without_volume = rf_without_volume.predict(
    test[features_without_current_volume]
)

ablation_results = pd.DataFrame([
    regression_metrics(
        "Random Forest — all selected features",
        y_test,
        test_predictions["Random Forest"],
    ),
    regression_metrics(
        "Random Forest — without current traffic",
        y_test,
        prediction_without_volume,
    ),
])

display(ablation_results.round(3))

## 17. Results summary

The following cell identifies the best machine-learning model by MAE and produces a short summary using the scores from this actual run.

Use the generated values when presenting the results. Do not describe R² as accuracy, and do not claim that a model is better than a baseline unless the displayed metrics support that conclusion.

In [ ]:
machine_learning_results = metrics_df[
    metrics_df["Model"].isin(["Decision Tree", "Random Forest"])
].sort_values("MAE")

best_model_name = machine_learning_results.iloc[0]["Model"]
best_result = machine_learning_results.iloc[0]

baseline_result = metrics_df[
    metrics_df["Model"] == "Persistence baseline"
].iloc[0]

mae_change = (
    (baseline_result["MAE"] - best_result["MAE"])
    / baseline_result["MAE"] * 100
)

print("=" * 64)
print("PROJECT RESULTS — NEXT-HOUR TRAFFIC VOLUME PREDICTION")
print("=" * 64)
print(f"Best machine-learning model by MAE: {best_model_name}")
print(f"MAE : {best_result['MAE']:.2f} vehicles/hour")
print(f"RMSE: {best_result['RMSE']:.2f} vehicles/hour")
print(f"R²  : {best_result['R2']:.4f}")

if mae_change > 0:
    print(f"The best ML model reduced MAE by {mae_change:.2f}% vs persistence.")
elif mae_change < 0:
    print(f"The best ML model had {abs(mae_change):.2f}% higher MAE than persistence.")
else:
    print("The best ML model matched the persistence baseline MAE.")

print(f"Training observations: {len(train):,}")
print(f"Testing observations: {len(test):,}")
print(f"Test period: {test['next_time'].min()} to {test['next_time'].max()}")
print("=" * 64)

## 18. Conclusion

This project developed an end-to-end machine-learning workflow for estimating the number of vehicles recorded in the next hour on the westbound I-94 study location. The workflow included data inspection, timestamp consolidation, exploratory analysis, next-hour target creation, chronological model evaluation, baseline comparisons, error analysis, and feature-importance inspection.

### What to discuss from the results

When presenting the notebook, the team should explain:

1. Which model achieved the lowest MAE on the held-out future period?
2. Did it outperform the persistence baseline?
3. How did it compare with the same-hour-yesterday baseline?
4. What do the actual-versus-predicted plot and residual plot reveal?
5. What happened when current traffic volume was removed?

### Limitations

- The data represents one westbound I-94 study location; findings may not generalise to other roads or cities.
- The observations are historical (2012–2018), so present-day traffic conditions may differ.
- The model uses current-hour weather observations. A deployed one-hour-ahead system would need weather forecasts available at prediction time.
- Traffic volume is not the same as congestion. The dataset does not provide official congestion labels, vehicle speeds, or road density.
- Evaluation uses one chronological 80/20 holdout. Rolling-window validation would provide a stronger assessment.
- Feature importance shows model behaviour, not causal relationships.

### Final conclusion

The project demonstrates how historical traffic and weather observations can be transformed into a supervised regression problem and evaluated using time-aware testing. Whether machine learning offers a practical improvement over simple baselines must be judged from the measured test results above.

## 19. Questions to prepare for the project presentation

**1. Why is this a regression problem?**  
Because the target is a numeric count of vehicles in the next hour, not a class label.

**2. What is the target variable?**  
`target_next_hour`, created from the traffic volume at the next consecutive hourly timestamp.

**3. Why are repeated timestamps consolidated?**  
To ensure there is one observation per timestamp before constructing the next-hour target.

**4. Why use a chronological train-test split?**  
Because the task is to predict future observations. Training on later dates and testing on earlier dates would not represent the intended forecasting setup.

**5. What is the persistence baseline?**  
It assumes that next-hour traffic volume will equal the current-hour traffic volume.

**6. What is the difference between MAE and RMSE?**  
Both measure prediction error. RMSE penalises large errors more strongly, while MAE is the average absolute error.

**7. Is R² the model's accuracy?**  
No. R² is a regression goodness-of-fit measure, not classification accuracy.

**8. Does the project detect congestion?**  
No. It predicts vehicle volume. Congestion detection would require suitable congestion labels or measurements such as speed and density.

**9. Why compare with the same hour yesterday?**  
Because traffic can follow daily patterns, so it is useful to compare machine learning with a simple daily-pattern baseline.

**10. What would improve the project next?**  
Use more recent data, test multiple roads, add speed or density measurements, use forecast weather data, and apply rolling-window validation.

---

## References

1. Hogue, J. (2019). *Metro Interstate Traffic Volume* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5X60B.
2. scikit-learn documentation: https://scikit-learn.org/stable/
3. pandas documentation: https://pandas.pydata.org/docs/
4. Matplotlib documentation: https://matplotlib.org/stable/

**Dataset licence:** Creative Commons Attribution 4.0 International (CC BY 4.0).